<a href="https://colab.research.google.com/github/robertrichard86/Kaggle/blob/main/Titanic_com_Optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Carregamento e Preparação dos Dados
Importação das bibliotecas e carregamento dos arquivos brutos do Kaggle, preservando os IDs para a submissão.

In [2]:
import pandas as pd
import numpy as np
import re
!pip install optuna
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

#Dados do Kaggle: https://www.kaggle.com/competitions/titanic/data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 8.0 MB/s eta 0:00:00


In [4]:
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')
test_passenger_id = test_data['PassengerId']

## 2. Engenharia de Atributos (Feature Engineering)
Criação de novas variáveis como Title (Título extraído do nome), FamilySize e Deck. Estas colunas ajudam o modelo a entender padrões de sobrevivência baseados em status e localização no navio.

In [5]:
def extract_features(df):
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss').replace('Ms', 'Miss').replace('Mme', 'Mrs')
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = 0
    df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1
    df['Cabin'] = df['Cabin'].fillna('U')
    df['Deck'] = df['Cabin'].map(lambda x: re.compile("([a-zA-Z]+)").search(x).group())
    return df

train_data = extract_features(train_data)
test_data = extract_features(test_data)

<>:3: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_184/2190567917.py:3: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


## 3. Tratamento de Valores Nulos
Imputação inteligente de dados. A idade é preenchida conforme a mediana de cada título, evitando distorções, enquanto as demais colunas são tratadas por moda ou mediana.

In [6]:
train_data['Age'] = train_data['Age'].fillna(train_data.groupby('Title')['Age'].transform('median'))
test_data['Age'] = test_data['Age'].fillna(test_data.groupby('Title')['Age'].transform('median'))
train_data['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)
test_data['Fare'].fillna(test_data['Fare'].median(), inplace=True)

/tmp/ipykernel_184/4058237710.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_data['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)
/tmp/ipykernel_184/4058237710.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(

## 4. Pré-processamento e Encoding
Conversão de variáveis categóricas em numéricas via One-Hot Encoding e alinhamento das colunas entre os datasets de treino e teste.

In [9]:
cols_to_drop = ['Name', 'Ticket', 'Cabin', 'PassengerId']
train_data.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_data.drop(columns=cols_to_drop, inplace=True, errors='ignore')

categorical_cols = ['Sex', 'Embarked', 'Pclass', 'Title', 'Deck']
train_data = pd.get_dummies(train_data, columns=categorical_cols, drop_first=True)
test_data = pd.get_dummies(test_data, columns=categorical_cols, drop_first=True)

features = train_data.drop(columns=['Survived']).columns
test_data = test_data.reindex(columns=features, fill_value=0)

X = train_data[features]
y = train_data['Survived']

## 5. Otimização Bayesiana (Optuna) e Treino Final
Uso do framework Optuna para testar automaticamente as melhores combinações de hiperparâmetros. Após encontrar a configuração ideal, o modelo final é treinado sobre todo o conjunto de dados.

In [10]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy'])
    }
    clf = RandomForestClassifier(**params, random_state=42)
    return cross_val_score(clf, X, y, n_jobs=-1, cv=5).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

model = RandomForestClassifier(**study.best_params, random_state=42)
model.fit(X, y)
print(f"Melhor acurácia na validação: {study.best_value:.4f}")

[I 2026-03-04 12:58:28,539] A new study created in memory with name: no-name-7ec2fa5c-d585-47e1-a46e-e1bbb26579c8
[I 2026-03-04 12:58:48,979] Trial 0 finished with value: 0.8237900947837551 and parameters: {'n_estimators': 704, 'max_depth': 12, 'min_samples_split': 14, 'min_samples_leaf': 6, 'criterion': 'gini'}. Best is trial 0 with value: 0.8237900947837551.
[I 2026-03-04 12:58:53,451] Trial 1 finished with value: 0.8136651810934655 and parameters: {'n_estimators': 550, 'max_depth': 3, 'min_samples_split': 13, 'min_samples_leaf': 6, 'criterion': 'gini'}. Best is trial 0 with value: 0.8237900947837551.
[I 2026-03-04 12:58:55,930] Trial 2 finished with value: 0.8282844768062269 and parameters: {'n_estimators': 267, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 4, 'criterion': 'entropy'}. Best is trial 2 with value: 0.8282844768062269.
[I 2026-03-04 12:59:01,410] Trial 3 finished with value: 0.8204193082669011 and parameters: {'n_estimators': 416, 'max_depth': 11, 'min_sa

Melhor acurácia na validação: 0.8328


## 6. Geração da Submissão
Predição dos resultados sobre o conjunto de teste e exportação para o arquivo submission.csv no formato padrão da competição.

In [11]:
test_predictions = model.predict(test_data)
output = pd.DataFrame({'PassengerId': test_passenger_id, 'Survived': test_predictions})
output.to_csv('submission.csv', index=False)
print("Arquivo 'submission.csv' pronto!")

Arquivo 'submission.csv' pronto!
